<a href="https://colab.research.google.com/github/Sabique-Islam/view/blob/main/ViewV0_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# View
text-to-video pipeline using CogVideoX-2B, designed to run on a standard T4 GPU in Google Colab.

In [ ]:
!pip install -q diffusers transformers accelerate imageio[ffmpeg] sentencepiece
!pip install -q git+https://github.com/huggingface/diffusers.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
import gc
from diffusers import CogVideoXPipeline, CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video
import imageio
import numpy as np
from google.colab import files
import os

# Create output directory
OS_OUTPUT_PATH = "/content/output"
os.makedirs(OS_OUTPUT_PATH, exist_ok=True)

# Check for GPU availability
if not torch.cuda.is_available():
    print("Go to 'Runtime' -> 'Change runtime type' and select 'T4 GPU'")
else:
    # Load CogVideoX-2B model
    # According to latest docs, we use float16 for T4 efficiency
    pipe = CogVideoXPipeline.from_pretrained(
        "THUDM/CogVideoX-2b",
        torch_dtype=torch.float16
    )

    # sequential_cpu_offload is the most memory-efficient for 16GB VRAM
    pipe.enable_sequential_cpu_offload()
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()

    print("Model loaded successfully with Sequential CPU Offloading!")

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Model loaded successfully with Sequential CPU Offloading!


## Phase 3: Shot Planning
We will use a simple LLM-like logic (using a template) to break down a prompt into a shot list.

In [ ]:
import json

def plan_shots(user_input):
    # In production, we will call a text-LLM here.
    # But here lets simulate the LLM output for the specific prompt.
    # Prompt: "Explain TCP three-way handshake"

    shot_list = [
        {
            "shot": 1,
            "description": "A digital network interface with a glowing 'SYN' packet traveling across a blue circuit board.",
            "camera": "Static close-up",
            "duration": 3
        },
        {
            "shot": 2,
            "description": "Two servers glowing in a dark room, a 'SYN-ACK' signal bouncing back with neon light trails.",
            "camera": "Slow zoom out",
            "duration": 3
        },
        {
            "shot": 3,
            "description": "The connection is established; a green 'ACK' signal completes the circuit, data flows rapidly.",
            "camera": "Cinematic pan",
            "duration": 3
        }
    ]
    return shot_list

user_prompt = "Explain TCP three-way handshake"
shots = plan_shots(user_prompt)
print(json.dumps(shots, indent=2))

[
  {
    "shot": 1,
    "description": "A digital network interface with a glowing 'SYN' packet traveling across a blue circuit board.",
    "camera": "Static close-up",
    "duration": 3
  },
  {
    "shot": 2,
    "description": "Two servers glowing in a dark room, a 'SYN-ACK' signal bouncing back with neon light trails.",
    "camera": "Slow zoom out",
    "duration": 3
  },
  {
    "shot": 3,
    "description": "The connection is established; a green 'ACK' signal completes the circuit, data flows rapidly.",
    "camera": "Cinematic pan",
    "duration": 3
  }
]


In [ ]:
import gc
import torch
import os

clip_paths = []

for i, shot in enumerate(shots):
    print(f"\n--- Processing Shot {shot['shot']}/{len(shots)} ---")
    print(f"Prompt: {shot['description']}")

    # Generate video frames
    with torch.no_grad():
        video_generate = pipe(
            prompt=shot['description'],
            num_videos_per_prompt=1,
            num_inference_steps=50,
            num_frames=49,
            use_dynamic_cfg=True,
            guidance_scale=6.0,
            generator=torch.Generator(device='cpu').manual_seed(42)
        ).frames[0]

    path = os.path.join(OS_OUTPUT_PATH, f'shot_{i}.mp4')
    export_to_video(video_generate, path, fps=8)
    clip_paths.append(path)

    # Cleanup VRAM
    del video_generate
    gc.collect()
    torch.cuda.empty_cache()


--- Processing Shot 1/3 ---
Prompt: A digital network interface with a glowing 'SYN' packet traveling across a blue circuit board.


  0%|          | 0/50 [00:00<?, ?it/s]


--- Processing Shot 2/3 ---
Prompt: Two servers glowing in a dark room, a 'SYN-ACK' signal bouncing back with neon light trails.


  0%|          | 0/50 [00:00<?, ?it/s]


--- Processing Shot 3/3 ---
Prompt: The connection is established; a green 'ACK' signal completes the circuit, data flows rapidly.


  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
if clip_paths:
    # Create the concat list file with absolute paths
    with open("list.txt", "w") as f:
        for p in clip_paths:
            if os.path.exists(p):
                f.write(f"file '{os.path.abspath(p)}'\n")

    # Stitching using FFmpeg with re-encoding to ensure compatibility
    !ffmpeg -y -f concat -safe 0 -i list.txt -c:v libx264 -pix_fmt yuv420p /content/output/final_video.mp4

    from IPython.display import HTML, display
    from base64 import b64encode

    def show_video(video_path):
        if os.path.exists(video_path):
            mp4 = open(video_path,'rb').read()
            data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
            return HTML(f"<video width=640 controls autoplay loop><source src='{data_url}' type='video/mp4'></video>")
        return "Error: Video file not found."

    display(show_video("/content/output/final_video.mp4"))
else:
    print("No clips were generated to stitch.")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

## Image-to-Video Support

In [ ]:
import torch
import gc
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import load_image
from google.colab import files

if 'pipe' not in globals():
    print("Please run Cell 2 first to load the base components.")
else:
    print("Initializing Image-to-Video Pipeline...")
    # Reuse components from the base pipe to avoid redundant memory usage
    i2v_pipe = CogVideoXImageToVideoPipeline.from_pretrained(
        "THUDM/CogVideoX-2b",
        transformer=pipe.transformer,
        vae=pipe.vae,
        scheduler=pipe.scheduler,
        tokenizer=pipe.tokenizer,
        text_encoder=pipe.text_encoder,
        torch_dtype=torch.float16
    )

    i2v_pipe.enable_sequential_cpu_offload()
    i2v_pipe.vae.enable_slicing()
    i2v_pipe.vae.enable_tiling()

    print("Please upload your image:")
    uploaded = files.upload()

    if uploaded:
        image_path = list(uploaded.keys())[0]
        image = load_image(image_path)

        print("Generating Video (this takes a looooooot of time)...")
        with torch.no_grad():
            video_frames = i2v_pipe(
                image=image,
                prompt="Cinematic movement, high quality, 4k, realistic texture.",
                num_inference_steps=50,
                num_frames=49, # Optimized for T4
                guidance_scale=6.0,
                generator=torch.Generator(device='cpu').manual_seed(42)
            ).frames[0]

        out_path = "/content/output/i2v_output.mp4"
        export_to_video(video_frames, out_path, fps=8)

        # Standard cleanup
        del video_frames
        gc.collect()
        torch.cuda.empty_cache()

        # Use existing show_video function from the stitching cell or define locally
        from IPython.display import HTML, display
        from base64 import b64encode
        mp4 = open(out_path,'rb').read()
        data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
        display(HTML(f'<video width=640 controls><source src="{data_url}" type="video/mp4"></video>'))

Initializing Image-to-Video Pipeline...


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Please upload your image:


KeyboardInterrupt: 